### Book Recomendation 


In [1]:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from collections import Counter
import joblib
import os
from PIL import Image, ImageDraw, ImageFont

In [2]:
# ===============================================================
# 🔹 Load & Clean Dataset
# ===============================================================

books = pd.read_csv(r"C:\Users\Shrey\OneDrive\Documents\book recommendation system\data(new)\book_dataset.csv")

books["title"] = books["title"].str.strip().str.lower()
books["author"] = books["author"].fillna("Unknown")
books["genres"] = books["genres"].fillna("")
books["description"] = books["description"].fillna("")
books.drop_duplicates(subset="title", inplace=True)

In [3]:
# Simulated ratings for demonstration
ratings = pd.DataFrame({
    "user_id": np.random.randint(1000, 1100, size=10000),
    "title": np.random.choice(books["title"], size=10000),
    "rating": np.random.randint(1, 6, size=10000)
})
ratings["title"] = ratings["title"].str.strip().str.lower()
ratings_with_books = ratings.merge(books, on="title")

In [4]:
# ===============================================================
# 🎭 Genre & Stats Processing
# ===============================================================
books["genres"] = books["genres"].fillna("").apply(lambda x: ", ".join([g.strip() for g in x.split(",") if g.strip()]))
all_genres = [genre.strip() for sublist in books["genres"].dropna().str.split(",") for genre in sublist]
genre_counts = Counter(all_genres)
top_genres = [genre for genre, count in genre_counts.most_common(30)]

# Add random rating metadata
books["ratingsCount"] = np.random.randint(10, 5000, size=len(books))
books["reviewsCount"] = np.random.randint(5, 1000, size=len(books))
books["rating"] = np.round(np.random.uniform(2.5, 5.0, size=len(books)), 1)

In [5]:
# ===============================================================
# 🖼️ Local Cover System
# ===============================================================
COVER_FOLDER = "book_covers"
os.makedirs(COVER_FOLDER, exist_ok=True)

def get_local_cover(title):
    """
    Checks if a cover image exists in book_covers/.
    If not, creates a placeholder image with the title written on it.
    Returns the path to the image file.
    """
    safe_name = (
        title.strip()
        .replace(" ", "_")
        .replace("/", "_")
        .replace(":", "")
        .replace("'", "")
    )
    file_path = os.path.join(COVER_FOLDER, f"{safe_name}.jpg")

    # ✅ Already exists
    if os.path.exists(file_path):
        return file_path

    # ⚠️ Create placeholder if missing
    img = Image.new("RGB", (400, 600), color=(200, 200, 200))
    draw = ImageDraw.Draw(img)

    try:
        font = ImageFont.truetype("arial.ttf", 24)
    except:
        font = ImageFont.load_default()

    # Wrap title text
    words = title.title().split()
    lines, line = [], ""
    for word in words:
        if len(line + word) < 18:
            line += word + " "
        else:
            lines.append(line.strip())
            line = word + " "
    lines.append(line.strip())

    y = 250
    for line in lines:
        text_width = draw.textlength(line, font=font)
        x = (400 - text_width) // 2
        draw.text((x, y), line, fill=(0, 0, 0), font=font)
        y += 35

    img.save(file_path)
    return file_path


def fetch_poster(book_list):
    """Returns list of local cover image paths for given books."""
    return [get_local_cover(title) for title in book_list]

In [6]:
# ===============================================================
# 🤝 Collaborative Filtering Model
# ===============================================================
active_users = ratings_with_books["user_id"].value_counts()
active_users = active_users[active_users >= 10].index
ratings_with_books = ratings_with_books[ratings_with_books["user_id"].isin(active_users)]

num_rating = ratings_with_books.groupby("title")["rating"].count().reset_index()
num_rating.rename(columns={"rating": "num_of_rating"}, inplace=True)

final_rating = ratings_with_books.merge(num_rating, on="title")
final_rating = final_rating[final_rating["num_of_rating"] >= 4.5]
final_rating.drop_duplicates(["user_id", "title"], inplace=True)

book_pivot = final_rating.pivot_table(columns="user_id", index="title", values="rating").fillna(0)
book_sparse = csr_matrix(book_pivot)

collab_model = NearestNeighbors(algorithm="brute")
collab_model.fit(book_sparse)

print("✅ Collaborative model trained successfully!")
print("Pivot shape:", book_pivot.shape)

✅ Collaborative model trained successfully!
Pivot shape: (17, 58)


In [7]:
print("Pivot shape:", book_pivot.shape)

Pivot shape: (17, 58)


In [8]:
# ===============================================================
# 🧠 Content-Based Model
# ===============================================================
books["metadata"] = books["author"] + " " + books["genres"] + " " + books["description"]

tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(books["metadata"])

content_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
indices = pd.Series(books.index, index=books["title"]).drop_duplicates()

In [9]:
# ===============================================================
# 💡 Hybrid Recommendation Function
# ===============================================================
def hybrid_recommend(book_name, n=5, genre_filter=None):
    book_name = book_name.strip().lower()
    recommended_books = []

    if book_name in book_pivot.index:
        book_id = np.where(book_pivot.index == book_name)[0][0]
        _, suggestion = collab_model.kneighbors(book_pivot.iloc[book_id, :].values.reshape(1, -1), n_neighbors=n+1)
        recommended_books = [book_pivot.index[i] for i in suggestion[0][1:]]
    elif book_name in indices:
        idx = indices[book_name]
        sim_scores = sorted(list(enumerate(content_sim[idx])), key=lambda x: x[1], reverse=True)[1:n+10]
        recommended_books = [books["title"].iloc[i] for i, _ in sim_scores]

    if genre_filter:
        recommended_books = [
            title for title in recommended_books
            if genre_filter.lower() in books.loc[books["title"] == title, "genres"].values[0].lower()
        ][:n]

    poster_paths = fetch_poster(recommended_books)
    return recommended_books, poster_paths

In [10]:
# ===============================================================
# 🧾 Example Usage
# ===============================================================
books_to_test = ["bitten", "the blue umbrella", "casino royale", "the gardener"]
recommendations, covers = hybrid_recommend("bitten")

print("\n🎯 Recommended Books & Their Covers:\n")
for b, c in zip(recommendations, covers):
    print(f"{b.title():35} --> {c}")


🎯 Recommended Books & Their Covers:

The Night Train At Deoli And Other Stories --> book_covers\the_night_train_at_deoli_and_other_stories.jpg
The Blue Umbrella                   --> book_covers\the_blue_umbrella.jpg
The Portable Beat Reader            --> book_covers\the_portable_beat_reader.jpg
A Letter Of Mary                    --> book_covers\a_letter_of_mary.jpg
The Gardener                        --> book_covers\the_gardener.jpg
The Portable Dorothy Parker         --> book_covers\the_portable_dorothy_parker.jpg
Diamonds Are Forever                --> book_covers\diamonds_are_forever.jpg
First Indian On The Moon            --> book_covers\first_indian_on_the_moon.jpg
Casino Royale                       --> book_covers\casino_royale.jpg
Unaccustomed Earth                  --> book_covers\unaccustomed_earth.jpg
Discontents: New Queer Writers      --> book_covers\discontents_new_queer_writers.jpg
The Ministry Of Utmost Happiness    --> book_covers\the_ministry_of_utmost_happiness.j

In [11]:
# ===============================================================
# 💾 Save Model Artifacts
# ===============================================================
os.makedirs("nartifacts", exist_ok=True)
joblib.dump(collab_model, "nartifacts/model.pkl")
joblib.dump(book_pivot.index.tolist(), "nartifacts/books_name.pkl")
joblib.dump(final_rating, "nartifacts/final_rating.pkl")
joblib.dump(book_pivot, "nartifacts/book_pivot.pkl")
joblib.dump(tfidf_matrix, "nartifacts/tfidf_matrix.pkl")
joblib.dump(content_sim, "nartifacts/content_sim.pkl")
joblib.dump(indices, "nartifacts/indices.pkl")
joblib.dump(books, "nartifacts/books_df.pkl")
joblib.dump(top_genres, "nartifacts/top_genres.pkl")

print("\n✅ All artifacts and local covers are ready in 'nartifacts' and 'book_covers' folders!")


✅ All artifacts and local covers are ready in 'nartifacts' and 'book_covers' folders!
